# Fiedler-vector gradient & branch detection

*Converted from the exploratory script `Main3.py`.*

*Archived source: `Main3.py` @ commit `c36a461` — restore with `git show c36a461:Main3.py`.*


## Goal

Locate the **branch points** of a plant and segment it into coherent parts by
studying how the *Fiedler vector* (the second smallest graph eigenvector)
varies along the plant.

- **For biologists:** along a single stem or branch, the Fiedler value changes
  smoothly; where a branch meets the stem, the value changes sharply. So the
  *local gradient* of the Fiedler vector is large at junctions — a geometric
  signature of the plant's architecture.
- **For computer scientists:** compute the sparse Laplacian of the Riemannian
  graph, extract the Fiedler vector with `eigsh`, estimate a per-node gradient
  over the graph's neighbours, then run **connectivity-constrained
  agglomerative clustering** so that the resulting segments are *connected*
  sub-graphs of the plant.

## Implementation overview

1. Build the Riemannian graph (as in notebook 2).
2. Build the sparse Laplacian and extract the Fiedler vector with
   `scipy.sparse.linalg.eigsh`.
3. Compute the **neighbourhood gradient** of the Fiedler vector at each point.
4. Run `AgglomerativeClustering` constrained to the graph's connectivity.
5. Visualise the Fiedler vector, its gradient, and the final segments.

> The original script used `open3d.read_point_cloud` (obsolete) on
> `Data/arabi_densep_clean_segm.ply`; we use the current `open3d` API on a
> shipped cloud, subsampled so the agglomerative clustering step stays quick.


In [ ]:
import numpy as np
import open3d as o3d
import networkx as nx
import scipy.sparse as spsp
import sklearn.cluster as sk
import matplotlib.pyplot as plt

import spectral_clustering.similarity_graph as SGk

### 1. Load and build the graph

We load a *Chenopodium album* cloud, subsample it to keep the agglomerative
clustering fast, and build its k-NN Riemannian graph.

In [ ]:
pcd = o3d.io.read_point_cloud("../data/chenos/cheno_3.ply")
points = np.asarray(pcd.points)

In [ ]:
# Subsample for notebook speed (agglomerative clustering is the heavy step).
rng = np.random.default_rng(0)
idx = rng.choice(points.shape[0], 8000, replace=False)
sub = points[idx]
subpcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(sub))
print("subsampled points:", sub.shape)

In [ ]:
r = 8
G = SGk.create_riemannian_graph(subpcd, method="knn", nearest_neighbors=r)
print("nodes:", G.number_of_nodes(), " edges:", G.number_of_edges())

### 2. Sparse Laplacian and the Fiedler vector

The weighted Laplacian of the graph is built with `networkx` and converted to
a sparse matrix. `eigsh` (Lanczos) with `sigma=0` returns the smallest
eigenvalues; the **second** smallest eigenvector is the **Fiedler vector** —
it varies monotonically along the plant and is the classic tool for finding
natural cuts in a graph.

In [ ]:
Lcsr = nx.laplacian_matrix(G, weight="weight").astype(float)

In [ ]:
k = 2
keigenval, keigenvec = spsp.linalg.eigsh(Lcsr, k=k, sigma=0, which="LM")
fiedler = np.asarray(keigenvec[:, 1])
print("Fiedler eigenvalue:", float(keigenval[1]))

### 3. Gradient of the Fiedler vector over the graph

For each point we take the **max minus min** of the Fiedler value over its
neighbours in the graph. Smooth regions (inside a stem) have a small gradient;
**junctions** (where a branch attaches) have a large gradient — these are the
branch points we are looking for.

In [ ]:
A = nx.adjacency_matrix(G)
vp2_matrix = A.copy()
vp2_matrix[A.nonzero()] = fiedler[A.nonzero()[1]]

In [ ]:
node_neighbor_max = np.array([vp2_matrix[node].data.max() for node in range(A.shape[0])])
node_neighbor_min = np.array([vp2_matrix[node].data.min() for node in range(A.shape[0])])
vp2grad = node_neighbor_max - node_neighbor_min

In [ ]:
print("gradient stats: min %.3f  max %.3f" % (vp2grad.min(), vp2grad.max()))

### 4. Visualise Fiedler vector and its gradient

On the left, the Fiedler value varies smoothly along the plant. On the right,
the gradient is large (bright) exactly where branches join the stem.

In [ ]:
fig = plt.figure(figsize=(13, 6))
for i, (vals, title, cmap) in enumerate([
        (fiedler, "Fiedler vector", "Spectral"),
        (vp2grad, "Gradient of Fiedler vector", "hot")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    sc = ax.scatter(sub[:, 0], sub[:, 1], sub[:, 2], c=vals, cmap=cmap, s=2)
    fig.colorbar(sc, ax=ax, shrink=0.6)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 5. Connectivity-constrained agglomerative clustering

`AgglomerativeClustering` merges points bottom-up. Passing the graph's
adjacency as `connectivity` forces every cluster to be a **connected**
sub-graph of the plant — so each resulting segment is a spatially contiguous
region (a stem piece or a branch), not a scattered set of points.

In [ ]:
clustering = sk.AgglomerativeClustering(
    metric="euclidean", connectivity=A, linkage="ward", n_clusters=11
).fit(vp2grad[:, np.newaxis])
labels = clustering.labels_
print("cluster sizes:", np.bincount(labels))

### 6. Visualise the segments

Each colour is one connected segment found by the algorithm. The cloud is
partitioned into contiguous pieces following the plant's architecture.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(sub[:, 0], sub[:, 1], sub[:, 2], c=labels, cmap="tab20", s=2)
fig.colorbar(sc, ax=ax, label="segment")
ax.set_title("Agglomerative clustering on Fiedler gradient")
plt.show()

## Takeaway

The **gradient of the Fiedler vector** is a powerful, purely geometric cue for
finding the branch points of a plant, and connectivity-constrained
agglomerative clustering turns it into contiguous segments. This is the direct
ancestor of the branch / semantic segmentation implemented in the packaged
pipeline — see the current end-to-end entry point `script/Main_workflow.py`.